In [11]:
import os

import numpy as np
import random
import matplotlib.pyplot as plt

In [12]:
X_DIR = "../data/processed/patches/X.npy"
Y_DIR = "../data/processed/patches/Y.npy"

X = np.load(X_DIR).astype(np.float32)
Y = np.load(Y_DIR).astype(np.float32)

print("X shape:", X.shape, "dtype:", X.dtype, "min/max:", X.min(), X.max())
print("Y shape:", Y.shape, "dtype:", Y.dtype, "min/max:", Y.min(), Y.max())

print("Foreground pixel ratio:", np.mean(Y))

X shape: (136, 256, 256, 1) dtype: float32 min/max: 0.0 1.0
Y shape: (136, 256, 256, 1) dtype: float32 min/max: 0.0 1.0
Foreground pixel ratio: 0.14651524


In [13]:
def augment_patch(image, mask):
    """
    Apply the same geometric augmentation to image and mask.
    Apply intensity augmentation only to image.
    """

    # Random horizontal flip
    if random.random() < 0.5:
        image = np.flip(image, axis=1)
        mask = np.flip(mask, axis=1)

    # Random vertical flip
    if random.random() < 0.5:
        image = np.flip(image, axis=0)
        mask = np.flip(mask, axis=0)

    # Random 90-degree rotation
    k = random.choice([0, 1, 2, 3])
    image = np.rot90(image, k)
    mask = np.rot90(mask, k)

    # Brightness shift — image only
    if random.random() < 0.5:
        shift = random.uniform(-0.08, 0.08)
        image = image + shift

    # Contrast adjustment — image only
    if random.random() < 0.5:
        factor = random.uniform(0.85, 1.15)
        image = (image - image.mean()) * factor + image.mean()

    # Clip image back to valid range
    image = np.clip(image, 0, 1)

    # Ensure mask stays binary
    mask = (mask > 0.5).astype(np.float32)

    return image.astype(np.float32), mask.astype(np.float32)

In [14]:
def augment_dataset(X_train, Y_train, augment_factor=3):
    """
    Creates augmented copies of the training dataset.

    augment_factor=3 means:
    original dataset + 3 augmented versions
    """

    X_aug = [X_train]
    Y_aug = [Y_train]

    for _ in range(augment_factor):
        augmented_images = []
        augmented_masks = []

        for img, msk in zip(X_train, Y_train):
            img_aug, msk_aug = augment_patch(img, msk)
            augmented_images.append(img_aug)
            augmented_masks.append(msk_aug)

        X_aug.append(np.array(augmented_images, dtype=np.float32))
        Y_aug.append(np.array(augmented_masks, dtype=np.float32))

    X_aug = np.concatenate(X_aug, axis=0)
    Y_aug = np.concatenate(Y_aug, axis=0)

    return X_aug, Y_aug

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Before augmentation:")
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)

X_train_aug, Y_train_aug = augment_dataset(
    X_train,
    Y_train,
    augment_factor=3
)

print("\nAfter augmentation:")
print("X_train_aug:", X_train_aug.shape)
print("Y_train_aug:", Y_train_aug.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)

Before augmentation:
X_train: (108, 256, 256, 1)
Y_train: (108, 256, 256, 1)
X_val: (28, 256, 256, 1)
Y_val: (28, 256, 256, 1)

After augmentation:
X_train_aug: (432, 256, 256, 1)
Y_train_aug: (432, 256, 256, 1)
X_val: (28, 256, 256, 1)
Y_val: (28, 256, 256, 1)


In [16]:
history = model.fit(
    X_train_aug,
    Y_train_aug,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

NameError: name 'model' is not defined

In [ ]:
idx = np.random.randint(0, len(X_train_aug))

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(X_train_aug[idx].squeeze(), cmap="gray")
plt.title("Augmented Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(Y_train_aug[idx].squeeze(), cmap="gray")
plt.title("Augmented Mask")
plt.axis("off")

plt.show()